# Разделы сайта «Терновое королевство»: типы заданий, их число и темы

Ноутбук для Google Colab. Он читает HTML-страницу тренажёра по математике и собирает три таблицы:

1. **Разделы** — тип, число заданий, что отрабатывается, темы теории.
2. **Типы заданий** — сколько разделов и заданий каждого типа.
3. **Темы** — в каких разделах тема отрабатывается и сколько там заданий.

Страница динамическая: текст сайта лежит внутри тега `<script>`. Поэтому `BeautifulSoup` находит скрипт, а данные из него достают регулярные выражения.

**Как запустить**

- Вариант А: вставьте в переменную `URL` raw-ссылку на HTML-файл из репозитория GitHub.
- Вариант Б: оставьте `URL` пустым и перетащите HTML-файл в панель «Файлы» слева.

Результат сохраняется в `ternovoe_razdely.xlsx`.

In [ ]:
# Вариант А: raw-ссылка на HTML-файл, например
# https://raw.githubusercontent.com/ЛОГИН/РЕПОЗИТОРИЙ/main/ternovoe-korolevstvo.html
URL = ""

In [ ]:
import re, glob
import requests
import pandas as pd
from bs4 import BeautifulSoup

if URL:                                      # вариант А: скачать страницу запросом
    r = requests.get(URL, headers={"User-Agent": "Mozilla/5.0 (учебный проект, Google Colab)"}, timeout=30)
    r.raise_for_status()
    r.encoding = "utf-8"
    page = r.text
    print("Страница скачана:", len(page), "символов")
else:                                        # вариант Б: HTML-файл из панели «Файлы» слева
    found = glob.glob("*.html")
    assert found, "HTML-файл не найден: перетащите его в панель «Файлы» слева или заполните URL"
    page = open(found[0], encoding="utf-8").read()
    print("Читаю файл:", found[0])

soup = BeautifulSoup(page, "html.parser")
script = max((s.get_text() for s in soup.select("script")), key=len)   # код игры

def block(name):
    """Текст объявления `const NAME=...;` из кода страницы."""
    start = script.index(f"const {name}=")
    return script[start:script.index(";\n", start)]

# Названия тем теории: {'mul': 'Что такое умножение', ...}
theory = dict(re.findall(r"(\w+):\{t:'([^']+)'", block("THEORY")))
# Какие темы отрабатывает каждый раздел: {'bags': ['mul', 'parts'], ...}
topics = {k: re.findall(r"'(\w+)'", v) for k, v in re.findall(r"(\w+):\[([^\]]*)\]", block("TH_ST"))}

rows = []
pattern = r"\{id:'(\w+)',stk:'[^']+',name:'([^']+)',sub:'([^']+)'(?:,tmul:\d+)?,run:([^\n]+?)\},?\n"
for n, (sid, name, about, run) in enumerate(re.findall(pattern, block("STATIONS") + "\n"), start=1):
    if "runQuiz" in run:                     # раздел с заданиями: число записано прямо в описании
        kind = "задания с проверкой"
        total = int(re.search(r"total:(\d+)", run).group(1))
    else:                                    # игра на скорость: число записано в функции игры
        kind = "игра на скорость"
        body = script[script.index(f"function {run}("):][:400]
        m = re.search(r"(?:TOTAL|GOAL)=(\d+)|facts\.length<(\d+)", body)
        total = int(m.group(1) or m.group(2)) if m else None
    rows.append({"№": n, "Раздел": name, "Тип": kind, "Заданий": total,
                 "Что отрабатывается": about, "Темы теории": [theory.get(t, t) for t in topics.get(sid, [])]})

sections = pd.DataFrame(rows)
sections["Заданий"] = sections["Заданий"].astype("Int64")

# 1) Разделы: тип, число заданий, темы
table1 = sections.assign(**{"Темы теории": sections["Темы теории"].str.join("; ")})
# 2) Типы заданий: сколько разделов и заданий
table2 = sections.groupby("Тип").agg(Разделов=("Раздел", "count"), Заданий=("Заданий", "sum")).reset_index()
# 3) Темы: в каких разделах отрабатываются и сколько там заданий
table3 = (sections.explode("Темы теории").groupby("Темы теории")
          .agg(Разделов=("Раздел", "count"), Заданий=("Заданий", "sum"), Разделы=("Раздел", ", ".join))
          .sort_values("Заданий", ascending=False).reset_index().rename(columns={"Темы теории": "Тема"}))

display(table1, table2, table3)

with pd.ExcelWriter("ternovoe_razdely.xlsx") as xl:
    table1.to_excel(xl, sheet_name="Разделы", index=False)
    table2.to_excel(xl, sheet_name="Типы заданий", index=False)
    table3.to_excel(xl, sheet_name="Темы", index=False)
print("Сохранено: ternovoe_razdely.xlsx")